# Understanding the digitisation of the ODD

In [1]:
%load_ext autoreload
%autoreload 2

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import uproot
import glob
import awkward as ak
import itertools
import yaml
import os
from tqdm import tqdm
from pathlib import Path
import atlasify as atl
atl.ATLAS = "ColliderML"

from utils import load_config_data, get_hist_data, load_root_file

**Points of comparison**
For a specific configuration of the simulation (pt = 150MeV, eta = [-4, 4], with neutrals, with secondaries):
- For ITk, go back to the root file to get particle list (particularly, low pt and neutrals)
- How many target particles (be very explicit and apples to apples)
- How many target hits vs eta (theory that high eta in ITk is giving many target hits)
- How many background hits (low pt primary, or secondary) vs eta
- How many hits per target and background particle
- How many neutral hits per neutral particle in ODD?

## Load ODD Data

In [13]:
base_dir = Path("/global/cfs/cdirs/m3443/usr/dtmurnane/Side_Work/ACTS/outputs/itk_comparison/odd_output_config_pt50_postpt0_eta-4.0_4.0_neutralsFalse_secondariesTruePU400/proc_0")
odd_particles = load_root_file(base_dir / "particles.root")
odd_hits = load_root_file(base_dir / "hits.root")
odd_measurements = load_root_file(base_dir / "measurements.root")

In [14]:
odd_hits = odd_hits[odd_hits.event_id == 0]
# odd_measurements = odd_measurements[odd_measurements.event_nr == 0]
odd_particles = odd_particles[odd_particles.event_id == 0]


In [15]:
odd_hits

,event_id,geometry_id,particle_id,tx,ty,tz,tt,tpx,tpy,tpz,...,deltapx,deltapy,deltapz,deltae,index,volume_id,boundary_id,layer_id,approach_id,sensitive_id
entry,,,,,,,,,,,,,,,,,,,,,
0,0,1152921779484754177,1774418255348236288,98.907616,-5.626409,-1515.599976,4934.176270,0.282959,-0.048121,-4.339431,...,0.000346,0.000470,0.000061,-0.000043,5,16,0,4,0,1
1,0,1152921779484754177,1751922247813103641,48.481640,-7.031428,-1515.599976,1994.596680,0.001068,-0.001195,-0.003068,...,-0.000157,0.000168,-0.000072,-0.000034,6,16,0,4,0,1
2,0,1152921779484754177,1751900255802097664,56.295464,2.489612,-1515.599976,1941.702637,1.244291,0.075016,-32.326977,...,0.000132,0.000309,0.000035,-0.000029,1,16,0,4,0,1
3,0,1152921779484754177,1751900255802163226,56.238960,2.525041,-1515.537476,1942.700562,-0.000014,-0.000011,-0.000047,...,0.000014,0.000011,0.000047,-0.000002,0,16,0,4,0,1
4,0,1152921779484754177,1747396656292167680,67.652687,0.239545,-1515.599976,-1977.502686,0.281430,-0.016194,-6.591853,...,-0.000016,0.000508,0.000040,-0.000042,3,16,0,4,0,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
369907,0,2161728645771608066,815151534282309719,351.863037,942.278503,3009.500000,2884.921631,0.004800,0.027659,0.067036,...,0.000225,-0.000584,0.000140,-0.000076,4,30,0,12,0,192
369908,0,2161728645771608066,603482351040790530,440.926910,955.377319,3009.500000,1842.003540,-0.282474,0.026241,0.449176,...,-0.000364,0.000119,-0.000352,-0.000098,12,30,0,12,0,192
369909,0,2161728645771608066,508906759570653186,417.509827,912.287476,3009.500000,3867.093994,-0.103912,0.477256,1.385382,...,0.000648,0.000063,-0.000059,-0.000081,15,30,0,12,0,192


In [17]:
odd_particles

event_id          particle_id  particle_type  process  \
entry subentry                                                          
0     0                0     4503599644147712             22        0   
      1                0     4503599660924928            211        0   
      2                0     4503599677702144           -211        0   
      3                0     4503599694479360           -211        0   
      4                0     4503599711256576            211        0   
...                  ...                  ...            ...      ...   
      59972            0  1805959944894152704             22        0   
      59973            0  1805961044422557696             22        0   
      59974            0  1805961044439334912             22        0   
      59975            0  1805962143967739904             22        0   
      59976            0  1805962143984517120             22        0   

                      vx        vy         vz           vt        px  \
entry subentry                                                         
0     0        -0.006872 -0.017536  87.842751 -1566.635620 -2.773058   
      1        -0.006872 -0.017536  87.842751 -1566.635620 -0.422970   
      2        -0.006872 -0.017536  87.842751 -1566.635620 -0.074253   
      3        -0.006872 -0.017536  87.842751 -1566.635620  0.763931   
      4        -0.006872 -0.017536  87.842751 -1566.635620  2.574264   
...                  ...       ...        ...          ...       ...   
      59972    -0.005763 -0.012564 -29.701593 -1591.839478  0.108903   
      59973    -0.005795 -0.012566 -29.679373 -1591.848511 -0.132658   
      59974    -0.005795 -0.012566 -29.679373 -1591.848511  0.019493   
      59975    -0.005810 -0.012585 -29.670807 -1591.839966 -0.015941   
      59976    -0.005810 -0.012585 -29.670807 -1591.839966 -0.120642   

                      py  ...  vertex_primary  vertex_secondary  particle  \
entry subentry            ...                                               
0     0         4.312414  ...               1                 0         1   
      1         0.379447  ...               1                 0         2   
      2         0.019923  ...               1                 0         3   
      3         0.115129  ...               1                 0         4   
      4         3.843607  ...               1                 0         5   
...                  ...  ...             ...               ...       ...   
      59972     0.269102  ...             401                15        98   
      59973     0.163282  ...             401                16        99   
      59974     0.093474  ...             401                16       100   
      59975     0.028042  ...             401                17       101   
      59976     0.067863  ...             401                17       102   

                generation  sub_particle  e_loss  total_x0  total_l0  \
entry subentry                                                         
0     0                  0             0     0.0       0.0       0.0   
      1                  0             0     0.0       0.0       0.0   
      2                  0             0     0.0       0.0       0.0   
      3                  0             0     0.0       0.0       0.0   
      4                  0             0     0.0       0.0       0.0   
...                    ...           ...     ...       ...       ...   
      59972              0             0     0.0       0.0       0.0   
      59973              0             0     0.0       0.0       0.0   
      59974              0             0     0.0       0.0       0.0   
      59975              0             0     0.0       0.0       0.0   
      59976              0             0     0.0       0.0       0.0   

                number_of_hits  outcome  
entry subentry                           
0     0                      0        0  
      1                      0        0  
      2               

In [51]:
hits_tree = uproot.open(base_dir / "hits.root")
hits_tree.keys()

['hits;4', 'hits;3']

In [52]:
hits_keys = hits_tree.keys()
cycles = [int(key.split(';')[1]) for key in hits_keys]
latest_key = hits_keys[cycles.index(max(cycles))]
print(latest_key)

hits;4


In [53]:
hits_tree[latest_key]

<TTree 'hits' (21 branches) at 0x7fda4847ec50>

In [54]:
hits_data = hits_tree[latest_key].arrays()
hits_data

<Array [{event_id: 0, ...}, {...}, ..., {...}] type='1928697 * {event_id: u...'>

In [55]:
hits_df = ak.to_dataframe(hits_data)
hits_df


,event_id,geometry_id,particle_id,tx,ty,tz,tt,tpx,tpy,tpz,...,deltapx,deltapy,deltapz,deltae,index,volume_id,boundary_id,layer_id,approach_id,sensitive_id
entry,,,,,,,,,,,,,,,,,,,,,
0,0,1152921779484754177,1774418255348236288,98.907616,-5.626409,-1515.599976,4934.176270,0.282959,-0.048121,-4.339431,...,0.000346,0.000470,0.000061,-0.000043,5,16,0,4,0,1
1,0,1152921779484754177,1751922247813103641,48.481640,-7.031428,-1515.599976,1994.596680,0.001068,-0.001195,-0.003068,...,-0.000157,0.000168,-0.000072,-0.000034,6,16,0,4,0,1
2,0,1152921779484754177,1751900255802097664,56.295464,2.489612,-1515.599976,1941.702637,1.244291,0.075016,-32.326977,...,0.000132,0.000309,0.000035,-0.000029,1,16,0,4,0,1
3,0,1152921779484754177,1751900255802163226,56.238960,2.525041,-1515.537476,1942.700562,-0.000014,-0.000011,-0.000047,...,0.000014,0.000011,0.000047,-0.000002,0,16,0,4,0,1
4,0,1152921779484754177,1747396656292167680,67.652687,0.239545,-1515.599976,-1977.502686,0.281430,-0.016194,-6.591853,...,-0.000016,0.000508,0.000040,-0.000042,3,16,0,4,0,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1928692,4,2161728645771608066,391813167849734162,370.698669,875.239075,3009.500000,6590.354492,0.148099,-0.089714,0.174659,...,0.000392,0.005360,-0.003826,-0.001103,4,30,0,12,0,192
1928693,4,2161728645771608066,135107989693595649,360.105743,870.972412,3009.500000,1597.395630,0.198922,0.273671,1.017762,...,-0.000406,-0.000552,0.000129,-0.000094,2,30,0,12,0,192
1928694,4,2161728645771608066,135107989693595650,323.879761,886.640259,3009.500000,1597.983521,0.032076,0.320091,0.956933,...,-0.000505,-0.000053,-0.000051,-0.000081,2,30,0,12,0,192


In [56]:
measurements_tree = uproot.open(base_dir / "measurements.root")
measurements_tree.keys()


['measurements;5', 'measurements;4']

In [57]:
measurements_keys = measurements_tree.keys()
print(measurements_keys)

['measurements;5', 'measurements;4']


In [58]:
cycles = [int(key.split(';')[1]) for key in measurements_keys]
latest_key = measurements_keys[cycles.index(max(cycles))]
print(latest_key)

measurements;5


In [59]:
measurements_tree[latest_key]

<TTree 'measurements' (33 branches) at 0x7fda68750400>

In [60]:
data = measurements_tree[latest_key].arrays()
data


<Array [{event_nr: 0, ...}, {...}, ..., {...}] type='1928697 * {event_nr: i...'>

In [64]:
# Get all columns except the jagged ones
regular_columns = [field for field in data.fields 
                  if 'var' not in str(data[field].type)]

# Convert only the regular columns to DataFrame
measurements_df = ak.to_dataframe(data[regular_columns])

In [65]:
measurements_df

,event_nr,volume_id,layer_id,surface_id,rec_loc0,rec_loc1,rec_time,var_loc0,var_loc1,var_time,...,true_y,true_z,true_incident_phi,true_incident_theta,residual_loc0,residual_loc1,residual_time,pull_loc0,pull_loc1,pull_time
entry,,,,,,,,,,,,,,,,,,,,,
0,0,16,4,1,5.618162,22.931358,4940.616211,0.000225,0.000225,625.0,...,-5.626409,-1515.599976,-1.559762,-1.505642,-0.008246,0.023741,6.439941,-0.549761,1.582718,0.257598
1,0,16,4,1,7.008811,-27.505487,1937.288940,0.000225,0.000225,625.0,...,-7.031428,-1515.599976,-1.227115,-1.262486,-0.022617,0.012873,-57.307739,-1.507823,0.858180,-2.292310
2,0,16,4,1,-2.476325,-19.703934,1946.085693,0.000225,0.000225,625.0,...,2.489612,-1515.599976,-1.573122,-1.532322,0.013287,0.000605,4.383057,0.885820,0.040309,0.175322
3,0,16,4,1,-2.533069,-19.758469,1985.323242,0.000225,0.000225,625.0,...,2.525041,-1515.537476,-1.343331,-1.865052,-0.008029,0.002569,42.622681,-0.535250,0.171280,1.704907
4,0,16,4,1,-0.259118,-8.358468,-1988.163940,0.000225,0.000225,625.0,...,0.239545,-1515.599976,-1.568378,-1.528130,-0.019573,-0.011154,-10.661255,-1.304860,-0.743612,-0.426450
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1928692,4,30,12,192,6.829640,NaN,NaN,0.005184,NaN,NaN,...,875.239075,3009.500000,0.793953,1.726063,0.080170,NaN,NaN,1.113468,NaN,NaN
1928693,4,30,12,192,-1.579864,NaN,NaN,0.005184,NaN,NaN,...,870.972412,3009.500000,1.486967,1.259952,-0.017257,NaN,NaN,-0.239680,NaN,NaN
1928694,4,30,12,192,-40.931625,NaN,NaN,0.005184,NaN,NaN,...,886.640259,3009.500000,1.661368,1.257808,0.075130,NaN,NaN,1.043479,NaN,NaN


In [63]:
# Print the type structure of the awkward array
print(data.type)

# Look at the first few entries in detail
for field in data.fields:
    print(f"{field}: {data[field].type}")

1928697 * {event_nr: int32, volume_id: int32, layer_id: int32, surface_id: int32, rec_loc0: float32, rec_loc1: float32, rec_time: float32, var_loc0: float32, var_loc1: float32, var_time: float32, clus_size: int32, channel_value: var * float32, channel_loc0: var * int32, clus_size_loc0: int32, channel_loc1: var * int32, clus_size_loc1: int32, true_loc0: float32, true_loc1: float32, true_phi: float32, true_theta: float32, true_qop: float32, true_time: float32, true_x: float32, true_y: float32, true_z: float32, true_incident_phi: float32, true_incident_theta: float32, residual_loc0: float32, residual_loc1: float32, residual_time: float32, pull_loc0: float32, pull_loc1: float32, pull_time: float32}
event_nr: 1928697 * int32
volume_id: 1928697 * int32
layer_id: 1928697 * int32
surface_id: 1928697 * int32
rec_loc0: 1928697 * float32
rec_loc1: 1928697 * float32
rec_time: 1928697 * float32
var_loc0: 1928697 * float32
var_loc1: 1928697 * float32
var_time: 1928697 * float32
clus_size: 1928697 * 

In [23]:
df = ak.to_dataframe(data)
df

,,event_nr,volume_id,layer_id,surface_id,rec_loc0,rec_loc1,rec_time,var_loc0,var_loc1,var_time,...,true_y,true_z,true_incident_phi,true_incident_theta,residual_loc0,residual_loc1,residual_time,pull_loc0,pull_loc1,pull_time
entry,subentry,,,,,,,,,,,,,,,,,,,,,
